- https://realpython.com/python-send-email/

### Introduction:
- Sometimes we want to
    - receive email reminders from our code, 
    - send a confirmation email to users when they create an account, or 
    - send emails to members of our organization to remind them to pay their dues. 
- Sending emails manually is a time-consuming and error-prone task, but it’s easy to automate with Python.


#### What we'll learn:
- Set up a secure connection using SMTP_SSL() and .starttls()
- Use Python’s built-in smtplib library to send basic emails
- Send emails with HTML content and attachments using the email package
- Send multiple personalized emails using a CSV file with contact data
- Use the Yagmail package to send email through our Gmail account

## Setup Secure connection:
- Python comes with the built-in smtplib module for sending emails using the Simple Mail Transfer Protocol (SMTP). 
    - smtplib uses the RFC 821 protocol for SMTP. 
- The examples in this tutorial will use the Gmail SMTP server to send emails, but the same principles apply to other email services. 
- Although the majority of email providers use the same connection ports as the ones in this tutorial, you can run a quick Google search to confirm yours.
<br/><br/>
- To get started with this tutorial, set up a Gmail account for development, or set up an SMTP debugging server that discards emails you send and prints them to the command prompt instead. 
- A local SMTP debugging server can be useful for fixing any issues with email functionality and ensuring your email functions are bug-free before sending out any emails.

#### Option 1: Setting up a Gmail Account for Development
- If we want to use a Gmail account to send your emails, it's highly recommend setting up a throwaway account for the development of your code. 
- This is because we’ll have to adjust your Gmail account’s security settings to allow access from our Python code, and because there’s a chance we might accidentally expose our login details. 
- Also, it is found that the inbox of testing account rapidly filled up with test emails, which is reason enough to set up a new Gmail account for development.
<br/><br/>
- A nice feature of Gmail is that you can use the + sign to add any modifiers to your email address, right before the @ sign. 
    - For example, mail sent to sender+person1@example.com and sender+person2@example.com will both arrive at sender@example.com. 
- When testing email functionality, we can use this to emulate multiple addresses that all point to the same inbox.

- To set up a Gmail address for testing your code, do the following:
    - Create a new Google account: <https://accounts.google.com/signup/v2/webcreateaccount?flowName=GlifWebSignIn&flowEntry=SignUp>
    - Turn Allow less secure apps to ON. <https://myaccount.google.com/lesssecureapps>
        - Be aware that this makes it easier for others to gain access to your account.
        
        

> 🔴 **A credential was removed from this cell.**
> The original contained a real Gmail address and password in plain text.
> Never put credentials in a notebook - they end up in version control, and
> deleting them later does **not** remove them from the history.
> Read secrets from the environment instead, as **10.2** shows.

### Option 2: Setting up a Local SMTP Server
- We can test email functionality by running a local SMTP debugging server, using the smtpd module that comes pre-installed with Python. 
- Rather than sending emails to the specified address, it discards them and prints their content to the console. 
- Running a local debugging server means it’s not necessary to deal with encryption of messages or use credentials to log in to an email server.
- We can start a local SMTP debugging server by typing the following in Command Prompt:
    - `$ python -m smtpd -c DebuggingServer -n localhost:1025`

### Sending a Plain-Text Email
- . These are emails that you could write up in a simple text editor. There’s no fancy stuff like text formatting or hyperlinks.


#### Starting a Secure SMTP Connection
- When you send emails through Python, you should make sure that your SMTP connection is encrypted, so that your message and login credentials are not easily accessed by others.
    - **SSL (Secure Sockets Layer) and TLS (Transport Layer Security)** are two protocols that can be used to encrypt an SMTP connection. 
        - It’s not necessary to use either of these when using a local debugging server.
<br/><br/>
- There are two ways to start a secure connection with your email server:
    - Start an SMTP connection that is secured from the beginning using SMTP_SSL().
    - Start an unsecured SMTP connection that can then be encrypted using .starttls().
- In both instances, Gmail will encrypt emails using TLS, as this is the more secure successor of SSL. 
- As per Python’s Security considerations, it is highly recommended that we use create_default_context() from the ssl module. 
    - This will load the system’s trusted CA certificates, enable host name checking and certificate validation, and try to choose reasonably secure protocol and cipher settings.
- smtplib is Python’s built-in module for sending emails to any Internet machine with an SMTP or ESMTP listener daemon.
    - We’ll see how to use SMTP_SSL() first, as it instantiates a connection that is secure from the outset and is slightly more concise than the .starttls() alternative. 
    - Keep in mind that Gmail requires that you connect to port 465 if using SMTP_SSL(), and to port 587 when using .starttls().

#### Option 1: Using SMTP_SSL()
- The code example below creates a secure connection with Gmail’s SMTP server, using the SMTP_SSL() of smtplib to initiate a TLS-encrypted connection. 
- The default context of ssl validates the host name and its certificates and optimizes the security of the connection. 
- Make sure to fill in your own email address instead of sender2@example.com:

In [ ]:
import smtplib, ssl

port = 465  # For SSL
password = input("Type your password and press enter: ")

# Create a secure SSL context
context = ssl.create_default_context()

with smtplib.SMTP_SSL("smtp.gmail.com", port, context=context) as server:
    server.login("sender2@example.com", password)
    # TODO: Send email here

- Using with smtplib.SMTP_SSL() as server: makes sure that the connection is automatically closed at the end of the indented code block. 
    - If port is zero, or not specified, .SMTP_SSL() will use the standard port for SMTP over SSL (port 465).
<br/><br/>
- It’s not safe practice to store your email password in your code, especially if you intend to share it with others. 
    - Instead, use input() to let the user type in their password when running the script, as in the example above. 
    - If you don’t want your password to show on your screen when you type it, you can import the getpass module and use .getpass() instead for blind input of your password.

In [ ]:
import smtplib, ssl
import getpass
port = 465  # For SSL
password = getpass.getpass("Type your password and press enter: ")

# Create a secure SSL context
context = ssl.create_default_context()

with smtplib.SMTP_SSL("smtp.gmail.com", port, context=context) as server:
    server.login("sender2@example.com", password)
    # TODO: Send email here

#### Option 2: Using .starttls()
- Instead of using .SMTP_SSL() to create a connection that is secure from the outset, we can create an unsecured SMTP connection and encrypt it using .starttls().
- To do this, create an instance of smtplib.SMTP, which encapsulates an SMTP connection and allows you access to its methods. 
    - It's recommend to define our SMTP server and port at the beginning of our script to configure them easily.
- The code snippet below uses the construction server = SMTP(), rather than the format with SMTP() as server: which we used in the previous example. 
    - To make sure that your code doesn’t crash when something goes wrong, put your main code in a try block, and let an except block print any error messages to stdout:

In [ ]:
import smtplib, ssl
smtp_server = "smtp.gmail.com"
port = 587  # For starttls
sender_email = "sender2@example.com"
import getpass
password = getpass.getpass("Type your password and press enter: ")

# Create a secure SSL context
context = ssl.create_default_context()

# Try to log in to server and send email
try:
    server = smtplib.SMTP(smtp_server,port)
    server.ehlo() # Can be omitted
    server.starttls(context=context) # Secure the connection
    server.ehlo() # Can be omitted
    server.login(sender_email, password)
    # TODO: Send email here
except Exception as e:
    # Print any error messages to stdout
    print(e)
finally:
    server.quit() 

- To identify ourself to the server, .helo() (SMTP) or .ehlo() (ESMTP) should be called after creating an .SMTP() object, and again after .starttls().
- This function is implicitly called by .starttls() and .sendmail() if needed, so unless we want to check the SMTP service extensions of the server, it is not necessary to use .helo() or .ehlo() explicitly.

#### Sending Your Plain-text Email
- After we initiated a secure SMTP connection using either of the above methods, we can send our email using .sendmail(), which pretty much does what it says on the tin:

In [ ]:
sender_email = "sender2@example.com"
receiver_email = "sender3@example.com"
message = """\
Subject: Hi there

This message is sent from Python."""
server.sendmail(sender_email, receiver_email, message)

- The message string starts with "Subject: Hi there" followed by two newlines (\n). 
    - This ensures Hi there shows up as the subject of the email, and the text following the newlines will be treated as the message body.
- The code example below sends a plain-text email using SMTP_SSL():

In [ ]:
import smtplib, ssl

port = 465  # For SSL
smtp_server = "smtp.gmail.com"
sender_email = "sender2@example.com" # Enter your address
receiver_email = "sender3@example.com" # Enter receiver address 

import getpass
password = getpass.getpass("Type your password and press enter: ")

message = """\
Subject: Hi there

This message is sent from Python."""

context = ssl.create_default_context()
with smtplib.SMTP_SSL(smtp_server, port, context=context) as server:
    server.login(sender_email, password)
    server.sendmail(sender_email, receiver_email, message)

- For comparison, here is a code example that sends a plain-text email over an SMTP connection secured with .starttls(). The server.ehlo() lines may be omitted, as they are called implicitly by .starttls() and .sendmail(), if required:

In [ ]:
import smtplib, ssl

port = 587  # For starttls
smtp_server = "smtp.gmail.com"
sender_email = "sender2@example.com" # Enter your address
receiver_email = "sender3@example.com" # Enter receiver address 

import getpass
password = getpass.getpass("Type your password and press enter: ")

message = """\
Subject: Hi there

This message is sent from Python."""

context = ssl.create_default_context()
with smtplib.SMTP(smtp_server, port) as server:
    server.ehlo()  # Can be omitted
    server.starttls(context=context)
    server.ehlo()  # Can be omitted
    server.login(sender_email, password)
    server.sendmail(sender_email, receiver_email, message)

In [ ]:
import smtplib, ssl
from email.mime.text import MIMEText

port = 465
smtp_server = "smtp.gmail.com"
sender_email = "sender2@example.com" # Enter your address
receiver_email = ["sender3@example.com", 'aditya.tripathi005@hmail.com'] # Enter receiver address 

import getpass
password = getpass.getpass("Type your password and press enter: ")

msg = MIMEText('Hi, how are you today?')
msg['Subject'] = 'Hello'
msg['From'] = sender_email
msg['To'] = ', '.join(receiver_email)

# Create secure connection with server and send email
context = ssl.create_default_context()

with smtplib.SMTP_SSL(smtp_server, port, context=context) as server:
    server.login(sender_email, password)
    server.sendmail(sender_email, receiver_email, msg.as_string())

### Check mail

In [ ]:
import time
from itertools import chain
import email
import imaplib

imap_ssl_host = 'imap.gmail.com'  # imap.mail.yahoo.com
imap_ssl_port = 993
username = 'USERNAME or EMAIL ADDRESS'
password = 'PASSWORD'

# Restrict mail search. Be very specific.
# Machine should be very selective to receive messages.
criteria = {
    'FROM':    'PRIVILEGED EMAIL ADDRESS',
    'SUBJECT': 'SPECIAL SUBJECT LINE',
    'BODY':    'SECRET SIGNATURE',
}
uid_max = 0


def search_string(uid_max, criteria):
    c = list(map(lambda t: (t[0], '"'+str(t[1])+'"'), criteria.items())) + [('UID', '%d:*' % (uid_max+1))]
    return '(%s)' % ' '.join(chain(*c))
    # Produce search string in IMAP format:
    #   e.g. (FROM "sender4@example.com" SUBJECT "abcde" BODY "123456789" UID 9999:*)


def get_first_text_block(msg):
    type = msg.get_content_maintype()

    if type == 'multipart':
        for part in msg.get_payload():
            if part.get_content_maintype() == 'text':
                return part.get_payload()
    elif type == 'text':
        return msg.get_payload()


server = imaplib.IMAP4_SSL(imap_ssl_host, imap_ssl_port)
server.login(username, password)
server.select('INBOX')

result, data = server.uid('search', None, search_string(uid_max, criteria))

uids = [int(s) for s in data[0].split()]
if uids:
    uid_max = max(uids)
    # Initialize `uid_max`. Any UID less than or equal to `uid_max` will be ignored subsequently.

server.logout()


# Keep checking messages ...
# I don't like using IDLE because Yahoo does not support it.
while 1:
    # Have to login/logout each time because that's the only way to get fresh results.

    server = imaplib.IMAP4_SSL(imap_ssl_host, imap_ssl_port)
    server.login(username, password)
    server.select('INBOX')

    result, data = server.uid('search', None, search_string(uid_max, criteria))

    uids = [int(s) for s in data[0].split()]
    for uid in uids:
        # Have to check again because Gmail sometimes does not obey UID criterion.
        if uid > uid_max:
            result, data = server.uid('fetch', uid, '(RFC822)')  # fetch entire message
            msg = email.message_from_string(data[0][1].decode("utf-8"))
            
            uid_max = uid
        
            text = get_first_text_block(msg)
            print('New message :::::::::::::::::::::')
            print(text)

    server.logout()
    time.sleep(1)

In [ ]:
from itertools import chain
import email
import imaplib
import sys
from datetime import datetime

params = {
    'imap_ssl_host':    'imap.yandex.ru',
    'imap_ssl_port':    993,
    'username':         'checked_email@your_domain',
    'password':         'password',
    'criteria':         {
                            'FROM': 'put_here_searched@email_address',
                            'SUBJECT': 'Undelivered Mail Returned to Sender'
                        },
    'uid_max':          0,
    'folder':           'Tracking',
    'start_date':       'SINCE 04-Jul-2019',
    'end_date':         'BEFORE 05-Jul-2019'
}

def search_string(uid_max, criteria):
    c = list(map(lambda t: (t[0], '"' + str(t[1]) + '"'), criteria.items())) + [('UID', '%d:*' % (uid_max + 1))]
    return '(%s)' % ' '.join(chain(*c))

def get_first_text_block(msg):
    type = msg.get_content_maintype()

    if type == 'multipart':
        for part in msg.get_payload():
            if part.get_content_maintype() == 'text':
                return bytes.decode(part.get_payload(decode=1))
    elif type == 'text':
        return bytes.decode(msg.get_payload(decode=1))

def get_email_address(msg):
    # need to be updated for other email search criteria
    index_start = msg.index('<')
    index_end = msg.index('>')
    str = msg[index_start+1:index_end]
    return str

def main(params):
    try:
        server = imaplib.IMAP4_SSL(params['imap_ssl_host'], params['imap_ssl_port'])
        server.login(params['username'], params['password'])
        server.select(params['folder'])
        if params['end_date'] == '':
            search_dates = '(' + params['start_date'] + ')'
        elif params['start_date'] == '':
            search_dates = '(' + params['end_date'] + ')'
        else:
            search_dates = '(' + params['start_date'] + ' ' + params['end_date'] + ')'

        result, data = server.uid('search', search_dates, search_string(params['uid_max'], params['criteria']))
    except:
        print("Unexpected error:", sys.exc_info()[0])
        return 1
    else:
        uids = [int(s) for s in data[0].split()]
        if not uids:
            print("Emails found by search criteria: 0")
            return 3
        print("Emails found by search criteria: ", len(uids))

        undelivered_email_list = list()
        for uid in uids:
            params['uid_max'] = max(uids)   # in case for while loop to search for new messages with sleep
            try:
                result, emailData = server.uid('fetch', str(uid), '(RFC822)')
                #result, emailData = server.fetch(str(uid), '(RFC822)')
            except:
                print("Unexpected error:", sys.exc_info()[0])
                return 2
            msg = email.message_from_string(emailData[0][1].decode("utf-8"))
            text = get_first_text_block(msg)
            undelivered_email_list.append(get_email_address(text))

        undelivered_email_set = set(undelivered_email_list)
        dlist = {}
        print('\nUndelivered emails:')
        for x in undelivered_email_set:
            dlist.update({x: undelivered_email_list.count(x)})
        sorted_dict = {r: dlist[r] for r in sorted(dlist, key=dlist.get, reverse=True)}
        for k, v in sorted_dict.items():
            print(k, ' : ', v)
    finally:
        if 'server' in locals():
            server.logout()
    return

if __name__ == '__main__':
    start_time = datetime.now()
    exitcode = main(params)
    end_time = datetime.now()
    print('\nDuration: {}'.format(end_time - start_time))
    exit(exitcode)

In [ ]:
import smtplib
import time
import imaplib
import email

ORG_EMAIL   = "@gmail.com"
FROM_EMAIL  = "yourEmailAddress" + ORG_EMAIL
FROM_PWD    = "yourPassword"
SMTP_SERVER = "imap.gmail.com"
SMTP_PORT   = 993

def read_email_from_gmail():
    try:
        mail = imaplib.IMAP4_SSL(SMTP_SERVER)
        mail.login(FROM_EMAIL,FROM_PWD)
        mail.select('inbox')

        type, data = mail.search(None, 'ALL')
        mail_ids = data[0]

        id_list = mail_ids.split()   
        first_email_id = int(id_list[0])
        latest_email_id = int(id_list[-1])


        for i in range(latest_email_id,first_email_id, -1):
            typ, data = mail.fetch(i, '(RFC822)' )

            for response_part in data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_string(response_part[1])
                    email_subject = msg['subject']
                    email_from = msg['from']
                    print 'From : ' + email_from + '\n'
                    print 'Subject : ' + email_subject + '\n'

    except Exception, e:
        print str(e)